# 1.Nvidia环境测试

In [1]:
!nvidia-smi
!nvcc --version

Wed Nov 18 07:04:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# P0: 实现CUDA Kernel
目标：让 GPU 并行计算两个长度为 1000 的向量之和，并验证结果正确。

### vec_add.cu

In [2]:
%%writefile vec_add.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

// a, b, c 都指向 GPU 内存；n 是元素个数
__global__ void vec_add(const float* a, const float* b, float* c, int n) {
    // 算出全局下标 i，做越界保护，完成 c[i] = a[i] + b[i]
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n){
        c[i] = a[i] + b[i];
    }
}

int main() {
    const int N = 1000;                      // 元素个数
    const size_t bytes = N * sizeof(float);  // 字节数 = 1000 * 4 = 4000

    // 1. host 端分配并赋值
    float* h_a = (float*)malloc(bytes);
    float* h_b = (float*)malloc(bytes);
    float* h_c = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = i; h_b[i] = 2.0f * i; }

    // 2. GPU 端分配（d_ 前缀表示 device）
    float *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, bytes);
    cudaMalloc((void**)&d_b, bytes);
    cudaMalloc((void**)&d_c, bytes);

    // 3. 输入从 host 拷到 device：参数顺序是（目的地, 来源, 字节数, 方向）
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // 4. 启动 kernel
    int threads = 256;
    //只用整数运算实现向上取整
    int blocks = (N + threads - 1) / threads;
    vec_add<<<blocks, threads>>>(d_a, d_b, d_c, N);
    cudaError_t err = cudaDeviceSynchronize();  // 等 kernel 跑完，并拿到错误码
    if (err != cudaSuccess) printf("CUDA error: %s\n", cudaGetErrorString(err));

    // 5. 结果从 device 拷回 host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // 6. 验证：和 CPU 算的结果比最大误差
    float max_err = 0.0f;
    for (int i = 0; i < N; i++)
        max_err = fmaxf(max_err, fabsf(h_c[i] - (h_a[i] + h_b[i])));
    printf("blocks = %d, max_err = %g, c[999] = %g\n", blocks, max_err, h_c[N - 1]);

    // 7. 释放
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

Writing vec_add.cu


In [3]:
!nvcc vec_add.cu -o vec_add && ./vec_add

blocks = 4, max_err = 0, c[999] = 2997


### bench_cublas.cu

In [4]:
%%writefile bench_cublas.cu
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>
#include <chrono>
#include <cuda_runtime.h>
#include <cublas_v2.h>

// 计时一次 SGEMM（N×N 乘 N×N），返回秒
double time_sgemm(
    cublasHandle_t h,
    int N,
    const float* dA,
    const float* dB,
    float* dC
) {
    const float alpha = 1.0f, beta = 0.0f;

    cudaDeviceSynchronize();
    auto t0 = std::chrono::steady_clock::now();
    // cuBLAS 按列主序存矩阵。我们用行主序，所以把 A、B 交换传入，
    // 这样按行主序读出来的 dC 正好是 A·B
    cublasSgemm(
        h,
        CUBLAS_OP_N, CUBLAS_OP_N,
        N, N, N,
        &alpha,
        dB, N,
        dA, N,
        &beta,
        dC, N
        );

    cudaDeviceSynchronize();
    auto t1 = std::chrono::steady_clock::now();
    return std::chrono::duration<double>(t1 - t0).count();
}

int main() {
    cublasHandle_t h;
    cublasCreate(&h);

    const int sizes[] = {512, 1024, 2048, 4096, 8192};
    const int warmup = 3, reps = 10;

    printf("%6s %12s %10s\n", "N", "median(ms)", "GFLOPS");
    for (int N : sizes) {
        size_t n_elem = (size_t)N * N;            // 元素个数
        size_t bytes  = n_elem * sizeof(float);   // 字节数

        std::vector<float> hA(n_elem), hB(n_elem);
        for (size_t i = 0; i < n_elem; i++) {
            hA[i] = rand() / (float)RAND_MAX;
            hB[i] = rand() / (float)RAND_MAX;
        }

        float *dA, *dB, *dC;
        cudaMalloc((void**)&dA, bytes);
        cudaMalloc((void**)&dB, bytes);
        cudaMalloc((void**)&dC, bytes);
        cudaMemcpy(dA, hA.data(), bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(dB, hB.data(), bytes, cudaMemcpyHostToDevice);

        for (int i = 0; i < warmup; i++) time_sgemm(h, N, dA, dB, dC);

        std::vector<double> t(reps);
        for (int i = 0; i < reps; i++) t[i] = time_sgemm(h, N, dA, dB, dC);
        std::sort(t.begin(), t.end());

        double median = 0.0;
        if (reps % 2){
            median = t[reps / 2];
        }
        else{
            median = (t[reps / 2 - 1] + t[reps / 2]) / 2.0; // 单位：秒
        }

        double flops = 2.0 * N * N * N;
        double gflops = flops / median / 1e9;  // 用 N 和 median 算出 GFLOPS

        printf("%6d %12.3f %10.1f\n",
            N, median * 1e3, gflops);

        cudaFree(dA);
        cudaFree(dB);
        cudaFree(dC);
    }
    cublasDestroy(h);
    return 0;
}

Writing bench_cublas.cu


In [7]:
!nvcc bench_cublas.cu -o bench_cublas -lcublas && ./bench_cublas

     N   median(ms)     GFLOPS
   512        0.128     2090.2
  1024        0.749     2866.6
  2048        6.052     2838.7
  4096       31.012     4431.9
  8192      256.292     4290.1


预测效率有90%，实际效率只有51%，进行实验研究差异原因。



In [10]:
%%bash
nvidia-smi \
  --query-gpu=timestamp,clocks.sm,clocks.max.sm,power.draw,temperature.gpu,utilization.gpu \
  --format=csv -lms 100 > clocks.log &

monitor_pid=$!
trap 'kill "$monitor_pid" 2>/dev/null || true' EXIT

./bench_cublas

kill "$monitor_pid" 2>/dev/null || true
wait "$monitor_pid" 2>/dev/null || true

echo "=== 最后 20 个监控样本 ==="
tail -n 20 clocks.log

     N   median(ms)     GFLOPS
   512        0.085     3149.6
  1024        0.454     4735.0
  2048        3.213     5347.5
  4096       30.490     4507.6
  8192      253.146     4343.4
=== 最后 20 个监控样本 ===
2026/11/18 07:47:28.580, 870 MHz, 1590 MHz, 69.66 W, 44, 100 %
2026/11/18 07:47:28.692, 870 MHz, 1590 MHz, 69.76 W, 44, 100 %
2026/11/18 07:47:28.800, 870 MHz, 1590 MHz, 70.15 W, 44, 100 %
2026/11/18 07:47:28.912, 870 MHz, 1590 MHz, 69.96 W, 44, 100 %
2026/11/18 07:47:29.019, 870 MHz, 1590 MHz, 69.31 W, 44, 100 %
2026/11/18 07:47:29.128, 870 MHz, 1590 MHz, 69.47 W, 44, 100 %
2026/11/18 07:47:29.244, 855 MHz, 1590 MHz, 69.86 W, 44, 100 %
2026/11/18 07:47:29.352, 870 MHz, 1590 MHz, 69.27 W, 44, 100 %
2026/11/18 07:47:29.459, 855 MHz, 1590 MHz, 69.21 W, 44, 100 %
2026/11/18 07:47:29.569, 885 MHz, 1590 MHz, 68.98 W, 45, 100 %
2026/11/18 07:47:29.679, 870 MHz, 1590 MHz, 69.96 W, 45, 100 %
2026/11/18 07:47:29.786, 870 MHz, 1590 MHz, 69.37 W, 45, 100 %
2026/11/18 07:47:29.895, 870 MHz, 1590

In [11]:
!awk -F', ' '$4+0 > 30' clocks.log | head -n 30

2026/11/18 07:47:21.962, 1110 MHz, 1590 MHz, 39.31 W, 40, 23 %
2026/11/18 07:47:23.104, 915 MHz, 1590 MHz, 67.59 W, 41, 2 %
2026/11/18 07:47:23.221, 915 MHz, 1590 MHz, 70.05 W, 42, 93 %
2026/11/18 07:47:23.530, 1575 MHz, 1590 MHz, 31.45 W, 41, 6 %
2026/11/18 07:47:23.647, 1590 MHz, 1590 MHz, 32.93 W, 41, 6 %
2026/11/18 07:47:23.758, 1590 MHz, 1590 MHz, 33.23 W, 41, 0 %
2026/11/18 07:47:23.869, 1590 MHz, 1590 MHz, 33.13 W, 41, 0 %
2026/11/18 07:47:23.978, 1590 MHz, 1590 MHz, 33.13 W, 41, 0 %
2026/11/18 07:47:24.089, 1590 MHz, 1590 MHz, 33.33 W, 41, 0 %
2026/11/18 07:47:24.198, 1590 MHz, 1590 MHz, 33.33 W, 41, 0 %
2026/11/18 07:47:24.308, 1590 MHz, 1590 MHz, 33.43 W, 41, 0 %
2026/11/18 07:47:24.418, 1590 MHz, 1590 MHz, 33.43 W, 41, 0 %
2026/11/18 07:47:24.528, 1590 MHz, 1590 MHz, 33.61 W, 41, 0 %
2026/11/18 07:47:24.639, 1590 MHz, 1590 MHz, 33.21 W, 41, 0 %
2026/11/18 07:47:24.750, 1590 MHz, 1590 MHz, 33.13 W, 41, 0 %
2026/11/18 07:47:24.861, 1590 MHz, 1590 MHz, 33.23 W, 41, 0 %
2026/11/